In [ ]:
from pyspark.sql import SparkSession
from google.cloud import storage
import json

In [ ]:
'''
!mkdir -p "$HOME/spark-jars"

!curl -fL \
  "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar" \
  -o "$HOME/spark-jars/gcs-connector-hadoop3-latest.jar"
'''

In [ ]:
#!ls -lh "$HOME/spark-jars/gcs-connector-hadoop3-latest.jar"

In [ ]:
spark.conf.get(
    "spark.sql.adaptive.advisoryPartitionSizeInBytes"
)

In [ ]:
import os
from pyspark.sql import SparkSession

gcs_connector_jar = os.path.expanduser(
    "~/spark-jars/gcs-connector-hadoop3-latest.jar"
)

if not os.path.isfile(gcs_connector_jar):
    raise FileNotFoundError(
        f"Nie znaleziono konektora: {gcs_connector_jar}"
    )

spark = (
    SparkSession.builder
    .appName("development_for_silver_layer")
    .config(
        "spark.jars",
        gcs_connector_jar
    )
    .config(
        "spark.hadoop.fs.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem"
    )
    .config(
        "spark.hadoop.fs.AbstractFileSystem.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS"
    )
    .getOrCreate()
)

In [ ]:
spark = SparkSession \
        .builder\
        .appName("development_for_silver_layer") \
        .getOrCreate()

In [ ]:
spark.version

In [ ]:
storage_client = storage.Client()
bucket = storage_client.get_bucket("project-dev-storage")

In [ ]:
fil = json.loads(data_string)

#df = spark.createDataFrame(fil)
#df

In [ ]:
print(
    spark.sparkContext
    ._jsc
    .hadoopConfiguration()
    .get("fs.gs.impl")
)

In [ ]:
df_metadata = (
    spark.read
    .format("json")
    #.option("multiline", "true")
    .load("gs://project-dev-storage/company_forms/sec_metadata_folder/2026-08-06/")
)

In [ ]:
df_metadata.select(
    "cik",
    "name",
   # "entityType",
    "sicDescription",
    "fiscalYearEnd",
    "tickers",
    "exchanges"
    ).show()

In [ ]:
df_test = df_metadata_reports = df_metadata.select(
                        "filings.recent.accessionNumber"
)

df_test.show()

In [ ]:
df_test

In [ ]:
aqe_enabled = spark.conf.get("spark.sql.adaptive.enabled")

print(aqe_enabled)

In [ ]:
#df_test.write.format("delta").mode("overwrite").save("gs://project-dev-storage/company_forms/sec_metadata_folder/2026-08-06/test")
df_test.hint("REBALANCE").write.mode("overwrite").parquet("gs://project-dev-storage/company_forms/sec_metadata_folder/2026-08-06/test")

In [ ]:
''' 3 XBRL
df_content = (
    spark.read
    .format("json")
    .load("gs://project-dev-storage/company_forms/sec_facts_folder/2026-08-06/")
)
'''

In [ ]:
#df_temp = spark.read.json("gs://project-dev-storage/company_forms/sec_facts_folder/2026-08-06/0001045810_fact_file_2026-08-06 20:25:53.999746+02:00")
#df_temp.show()

In [ ]:
client = storage.Client()
blob = client.bucket("project-dev-storage").blob("company_forms/sec_metadata_folder/2026-08-06/0000320193_metadata_file_2026-08-06 20:25:53.999746+02:00")
json_content = blob.download_as_text(encoding="utf-8")

In [ ]:
json_file = json.loads(json_content)

In [ ]:
json_file["filings"]["recent"]["accessionNumber"]#.keys()

In [ ]:
'''
df_content.select(
        "cik",
        "entityName",
        "f
).show()
'''